# 🧪 종합 통합 테스트

전체 시스템 점검: PostgreSQL, Kafka, ErrorConsumer, Slack

In [ ]:
import sys
sys.path.insert(0, '../..')

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print("🧪 종합 통합 테스트 시작")
print("=" * 60)
print()

## 1️⃣ PostgreSQL 테스트

In [ ]:
print("\n[1] PostgreSQL 연결 테스트")
print("-" * 60)

import psycopg2
from config.settings import POSTGRES_HOST, POSTGRES_PORT, POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD

try:
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM raw_clickstream_events")
    count = cursor.fetchone()[0]
    cursor.close()
    conn.close()
    
    print(f"✅ PostgreSQL 연결 성공")
    print(f"   raw_clickstream_events: {count:,}개")
    postgres_ok = True
except Exception as e:
    print(f"❌ PostgreSQL 연결 실패: {e}")
    postgres_ok = False

## 2️⃣ Kafka 테스트

In [ ]:
print("\n[2] Kafka 연결 테스트")
print("-" * 60)

from kafka import KafkaConsumer, TopicPartition
from config.settings import KAFKA_BROKERS

try:
    consumer = KafkaConsumer(
        bootstrap_servers=','.join(KAFKA_BROKERS),
        enable_auto_commit=False
    )
    
    topics = consumer.topics()
    important_topics = ['clickstream', 'error_data']
    all_ok = all(t in topics for t in important_topics)
    
    print(f"✅ Kafka 연결 성공")
    for topic in important_topics:
        if topic in topics:
            partitions = consumer.partitions_for_topic(topic)
            print(f"   {topic:<15} : {len(partitions)}개 partition")
    
    consumer.close()
    kafka_ok = True
except Exception as e:
    print(f"❌ Kafka 연결 실패: {e}")
    kafka_ok = False

## 3️⃣ Error Consumer 테스트

In [ ]:
print("\n[3] Error Consumer 테스트")
print("-" * 60)

from src.kafka.error_consumer import ErrorDataConsumer

try:
    postgres_config = {
        'host': POSTGRES_HOST,
        'port': POSTGRES_PORT,
        'database': POSTGRES_DB,
        'user': POSTGRES_USER,
        'password': POSTGRES_PASSWORD
    }
    
    ec = ErrorDataConsumer(
        bootstrap_servers=','.join(KAFKA_BROKERS),
        group_id='test_consumer',
        postgres_config=postgres_config
    )
    
    # 에러 읽기 (최대 5개, 3초 대기)
    errors = ec.consume_errors(max_records=5, timeout_ms=3000)
    
    print(f"✅ Error Consumer 초기화 성공")
    print(f"   error_data topic: {len(errors)}개 메시지")
    
    if errors and len(errors) > 0:
        # 첫 번째 에러 표시
        err = errors[0]
        print(f"   샘플: {err['error_type']} ({err['status']})")
    
    ec.close()
    error_consumer_ok = True
except Exception as e:
    print(f"❌ Error Consumer 실패: {e}")
    error_consumer_ok = False

## 4️⃣ Slack Notifier 테스트

In [ ]:
print("\n[4] Slack Notifier 테스트")
print("-" * 60)

from src.utils.error_notifier import ErrorNotifier
from config.settings import SLACK_WEBHOOK_URL

try:
    notifier = ErrorNotifier(slack_webhook_url=SLACK_WEBHOOK_URL)
    
    # 간단한 테스트 알림
    success = notifier.send_success_alert(
        total_errors=0,
        stats={
            'critical_count': 0,
            'pending_count': 0,
            'by_error_type': {},
            'by_module': {}
        },
        channel='#kafka-error'
    )
    
    if success:
        print(f"✅ Slack 알림 발송 성공")
        print(f"   → #kafka-error 채널 확인")
        slack_ok = True
    else:
        print(f"⚠️  Slack 알림 발송 실패 (설정 확인)")
        slack_ok = False
except Exception as e:
    print(f"❌ Slack 초기화 실패: {e}")
    slack_ok = False

## 📊 최종 결과

In [ ]:
print("\n" + "=" * 60)
print("🧪 최종 테스트 결과")
print("=" * 60)

results = [
    ("PostgreSQL", postgres_ok),
    ("Kafka", kafka_ok),
    ("Error Consumer", error_consumer_ok),
    ("Slack Notifier", slack_ok)
]

for name, ok in results:
    status = "✅ PASS" if ok else "❌ FAIL"
    print(f"{name:<20} : {status}")

print()
all_pass = all(ok for _, ok in results)
if all_pass:
    print("✅ 모든 시스템이 정상입니다! 배포 준비 완료!")
else:
    print("⚠️  일부 시스템에 문제가 있습니다. 위의 로그를 확인하세요.")

print("=" * 60)